# features

> Continuous measurements for learned scoring

In [ ]:
#| default_exp features

In [ ]:
#| hide
from nbdev.showdoc import *

`features` measures prose for experiments with learned scoring. It returns rule-finding rates, sentence and paragraph statistics, grammatical counts, and measures of concrete detail and references.

Some measurements failed as standalone rules in the `para` and `score` notebooks. A model trained on labelled examples could still use them in combination. These experiments do not change slopometer's rule-based score.

In [ ]:
#| export
import statistics
from collections import Counter
from fastcore.utils import *
from slopometer.core import *
from slopometer.segment import *
from slopometer.lexicon import *
from slopometer.syntax import *
from slopometer.para import *
from slopometer.score import run_rules

In [ ]:
from fastcore.test import *
from nbdev.config import get_config

## Burstiness

The [Goh-Barabasi burstiness parameter](https://arxiv.org/abs/physics/0610233) is `(s - m) / (s + m)`, where `s` is the standard deviation and `m` the mean. Here it measures variation in sentence lengths. Uniform lengths give a value near -1. The value is zero when the standard deviation equals the mean and positive when it exceeds the mean.

Uniform sentence lengths can occur in both repetitive writing and clear reference prose. Burstiness alone is not a quality rule. `flattest_window` locates the consecutive sentences with the lowest length variation for closer review.

In [ ]:
#| export
def burstiness(xs):
    "Goh-Barabasi B over `xs`: -1 uniform, 0 random, positive bursty"
    if len(xs) < 2: return 0.0
    m, s = statistics.mean(xs), statistics.stdev(xs)
    return 0.0 if m + s == 0 else round((s - m) / (s + m), 3)

def flattest_window(
    xs, # Sentence lengths, in document order
    k=5, # Window size in sentences
):
    "`(start, spread)` of the length-`k` run with the lowest variation, for localizing monotone stretches"
    if len(xs) <= k: return 0, round(statistics.stdev(xs), 2) if len(xs) > 1 else 0.0
    spans = [(i, round(statistics.stdev(xs[i:i+k]), 2)) for i in range(len(xs) - k + 1)]
    return min(spans, key=lambda t: t[1])

In [ ]:
uniform = [10, 10, 11, 10, 10, 11, 10]
mixed = [3, 24, 8, 31, 5, 19, 12]
test_eq(burstiness(uniform) < -0.7, True)
test_eq(burstiness(mixed) > burstiness(uniform), True)
flattest_window([20, 21, 20, 3, 30, 7, 28, 4], k=3), round(burstiness(mixed), 2)

((0, 0.58), -0.17)

## The feature vector

`features` returns a flat dict with these groups of measurements:

- Weighted findings per rule and total density, per 100 alphabetic tokens in parsed prose. This denominator differs from `score_path`'s scored-word count.
- Sentence lengths, sentence-opening diversity, and sentences per paragraph.
- Adverb and adjective rates, noun-to-verb ratio, and the share of sentences whose root is "be".
- Numerals, proper nouns, and code spans per 100 words as a measure of specifics.
- The share of sentences with pronoun subjects and the share of definite nouns not previously mentioned.
- Rates of additive, causal, and adversative connectives.

The short keys serve as dataframe column names.

In [ ]:
#| export
_conn = dict(add={'also', 'furthermore', 'moreover', 'additionally'}, causal={'because', 'therefore', 'thus', 'hence'},
    advers={'but', 'however', 'although', 'though', 'yet'})

def features(txt):
    "Continuous measurements of markdown `txt`, one flat dict"
    blocks = segment(txt)
    docs = parse_blocks(blocks)
    prose = [d for d in docs if d is not None]
    sents = [s for d in prose for s in d.sents]
    toks = [t for d in prose for t in d]
    nw = max(sum(1 for t in toks if t.is_alpha), 1)
    res = dict(words=nw, sents=len(sents), headings=n_headings(blocks))
    agg = Counter()
    for f in run_rules(txt): agg[f.rule] += f.weight
    res |= {f'r_{k}': round(100*v/nw, 2) for k, v in agg.items()}
    res['density'] = round(100*sum(agg.values())/nw, 1)
    slens = [sum(1 for t in s if not t.is_punct) for s in sents]
    if len(slens) > 1:
        res |= dict(slen_mean=round(statistics.mean(slens), 1), slen_max=max(slens), slen_burst=burstiness(slens))
        starts = [next((t.lemma_.lower() for t in s if t.is_alpha), '') for s in sents]
        res['start_div'] = round(len(set(starts))/len(starts), 2)
    plens = [sum(1 for _ in d.sents) for d in prose]
    if plens: res |= dict(psents_mean=round(statistics.mean(plens), 1), psents_max=max(plens))
    pos = Counter(t.pos_ for t in toks)
    res |= dict(adv=round(100*pos['ADV']/nw, 1), adj=round(100*pos['ADJ']/nw, 1),
        noun_verb=round(pos['NOUN']/max(pos['VERB'], 1), 2),
        cop=round(sum(1 for s in sents if s.root.lemma_ == 'be')/max(len(sents), 1), 2))
    ncode = sum(len(re.findall(r'X{2,}', scrub(b.txt))) for b in blocks)
    res['specifics'] = round(100*(pos['NUM'] + pos['PROPN'] + ncode)/nw, 1)
    res['pron_subj'] = round(sum(1 for s in sents if any(t.dep_ in ('nsubj', 'nsubjpass') and t.pos_ == 'PRON'
        and t.head.dep_ == 'ROOT' for t in s))/max(len(sents), 1), 2)
    seen, ndef, ncold = set(), 0, 0
    for d in prose:
        for t in d:
            if t.pos_ not in ('NOUN', 'PROPN') or len(set(t.lower_)) == 1: continue
            lem = t.lemma_.lower()
            kids = [c.lower_ for c in t.children]
            if 'the' in kids and 'same' not in kids:
                ndef += 1
                if lem not in seen: ncold += 1
            seen.add(lem)
    res |= dict(defs=ndef, cold=round(ncold/max(ndef, 1), 2))
    res |= {f'conn_{k}': round(100*sum(1 for t in toks if t.lemma_.lower() in v)/nw, 2) for k, v in _conn.items()}
    return res

In [ ]:
sd = get_config().config_path/'samples'
texts = {f't{i}': (sd/f'theory{i}.md').read_text() for i in (1, 2, 3, 4)}
fx = {k: features(t) for k, t in texts.items()}
cols = ['density', 'slen_mean', 'slen_burst', 'start_div', 'psents_max', 'specifics', 'pron_subj', 'cold', 'cop']
test_eq(fx['t2']['specifics'] < fx['t4']['specifics'], True)
{k: {c: v[c] for c in cols if c in v} for k, v in fx.items()}

{'t1': {'density': 53.5,
  'slen_mean': 21.9,
  'slen_burst': -0.047,
  'start_div': 0.7,
  'psents_max': 5,
  'specifics': 3.4,
  'pron_subj': 0.09,
  'cold': 0.56,
  'cop': 0.35},
 't2': {'density': 5.4,
  'slen_mean': 10.9,
  'slen_burst': -0.336,
  'start_div': 0.48,
  'psents_max': 6,
  'specifics': 2.8,
  'pron_subj': 0.06,
  'cold': 0.46,
  'cop': 0.21},
 't3': {'density': 10.1,
  'slen_mean': 14.8,
  'slen_burst': -0.23,
  'start_div': 0.54,
  'psents_max': 7,
  'specifics': 2.5,
  'pron_subj': 0.03,
  'cold': 0.28,
  'cop': 0.09},
 't4': {'density': 1.5,
  'slen_mean': 12.4,
  'slen_burst': -0.26,
  'start_div': 0.61,
  'psents_max': 8,
  'specifics': 9.3,
  'pron_subj': 0.03,
  'cold': 0.3,
  'cop': 0.03}}

In these samples, `theory2` and `theory4` both have low rule scores. Their specifics rates differ: 2.8 versus 9.3. `theory2` also has the most uniform sentence lengths and the lowest opening-word diversity.

The share of sentences rooted in "be" falls from 0.35 to 0.21, 0.09, and 0.03 across the four samples. Pronoun-subject share also falls between the first and last samples. These observations suggest features to test on labelled data. They do not establish general scoring rules.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()